In [1]:
import pandas as pd
import numpy as np 
import nnresample 
import os 
from IPython.display import Audio
import pickle

## Open timit dataframes and get word speaker noise vocabulary

In [2]:
word_and_speaker_encodings = pickle.load( open( "/om4/group/mcdermott/user/jfeather/projects/model_metamers/figure_generation_notebooks/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']


In [3]:
path= '/om4/group/mcdermott/user/raygon/home/projects/public/jsinDataset/assets/data/interim/timit/'
timit_df = pd.read_pickle(os.path.join(path,'timitWordsDataframe_interim.pdpkl'))

In [4]:
timit_df.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,start_ms,end_ms
0,3050,5723,she,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2673,3050,41074,0.190625,0.357687
1,5723,10337,had,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4614,5723,36460,0.357687,0.646062
2,9190,11517,your,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2327,9190,35280,0.574375,0.719812
3,11517,16334,dark,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4817,11517,30463,0.719812,1.020875
4,16334,21199,suit,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4865,16334,25598,1.020875,1.324938


In [5]:
## All timit data 

padas_path = '/om4/group/mcdermott/user/raygon/home/projects/public/jsinDataset/assets/data/raw/timit/dataframes/'
pth = 'all_timit_data.pdpkl'

all_timit = pd.read_pickle(os.path.join(padas_path, pth))

In [6]:
all_timit.head()

,corpus,source,speaker_id,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,source_int,dialect_region_int,...,speaker_sex_int,sentence_type_int,sentence_id_int,sr,signal,phoneme,word,sentence,path,signal_length
0,timit,train-dr1-fcjf0-sa1,cjf0,f,sa,1,dr1,train,1680,0,...,0,0,0,16000,"[[3.051850947599719e-05, -3.051850947599719e-0...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,46797
1,timit,train-dr1-fcjf0-sa2,cjf0,f,sa,2,dr1,train,1681,0,...,0,0,1111,16000,"[[-3.051850947599719e-05, 0.0, 3.0518509475997...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,start_sa...,/home/raygon/deepFerret/data/sources/speech/ti...,34509
2,timit,train-dr1-fcjf0-si1027,cjf0,f,si,1027,dr1,train,1682,0,...,0,1,32,16000,"[[0.00024414807580797754, 0.000152592547379985...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,49460
3,timit,train-dr1-fcjf0-si1657,cjf0,f,si,1657,dr1,train,1683,0,...,0,1,731,16000,"[[0.00012207403790398877, 9.155552842799158e-0...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,45466
4,timit,train-dr1-fcjf0-si648,cjf0,f,si,648,dr1,train,1684,0,...,0,1,1952,16000,"[[-6.103701895199438e-05, 0.000183111056855983...",start_sample end_sample phoneme 0 ...,start_sample end_sample word 0 ...,...,/home/raygon/deepFerret/data/sources/speech/ti...,57856


### Get set of TIMIT words included in vocabulary

In [7]:
all_words = np.unique(np.concatenate([all_timit.word[ix].word.values for ix in range(all_timit.shape[0])]))

In [8]:
# get list of timit words included in model vocabulary 

valid_timit_words = all_words[np.in1d(all_words, list(class_map.values()))] 

### Need to parse through timit to get excerpts with valid words 

Here we parse the dataset for 2 coniditions:

1. the words occur such that we can make 2 second excerpts centered on the word
2. the word is in our vocabulary

For 1, we crudely make sure the word either overlaps the middle of the parent excerpt or that there is at least half a second context before word onset and after word offset (given a 2 second window). 


In [9]:
## Get good ixs by screening timing first then words 


def screen_row(row):
    mid_dur = row.signal_length//2
    mid_of_2 = row.start_sample >= 8000 and row.end_sample <= 24000
    mid_of_excerpt = row.start_sample <= mid_dur and row.end_sample > mid_dur
    long_enough = mid_of_2 or mid_of_excerpt 
    return long_enough 

good_timit_excerpts = timit_df[['start_sample', 'end_sample', 'word', 'signal_length']].apply(lambda x: screen_row(x), axis=1)

In [10]:
good_timit = timit_df[good_timit_excerpts].copy()

In [11]:
good_timit.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,start_ms,end_ms
2,9190,11517,your,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2327,9190,35280,0.574375,0.719812
3,11517,16334,dark,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4817,11517,30463,0.719812,1.020875
4,16334,21199,suit,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4865,16334,25598,1.020875,1.324938
5,21199,22560,in,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,1361,21199,24237,1.324938,1.410000
6,22560,28064,greasy,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,5504,22560,18733,1.410000,1.754000


In [12]:
# must be 2 seconds 

good_timit = good_timit[good_timit.signal_length >= 32000]

In [13]:
good_timit.shape

(18055, 17)

### Make new df with wanted attributes 

* updampled wav (16kHz -> 20kHz)
* word label
* speaker label
* speaker sex 
* speaker dialect

In [14]:
all_timit.columns

Index(['corpus', 'source', 'speaker_id', 'speaker_sex', 'sentence_type',
       'sentence_id', 'dialect_region', 'data_split', 'source_int',
       'dialect_region_int', 'data_split_int', 'speaker_id_int',
       'speaker_sex_int', 'sentence_type_int', 'sentence_id_int', 'sr',
       'signal', 'phoneme', 'word', 'sentence', 'path', 'signal_length'],
      dtype='object')

In [15]:
good_timit.columns

Index(['start_sample', 'end_sample', 'word', '_full_dataset_index', 'corpus',
       'path', 'source', 'speaker', 'sr', 'signal_length', 'duration_ms',
       'n_channels', 'word_length_samples', 'num_samples_before',
       'num_samples_after', 'start_ms', 'end_ms'],
      dtype='object')

In [16]:
munged_excerpt_labels = all_timit[['signal',
                                   'speaker_sex', 'sentence_type',
                                   'sentence_id', 'dialect_region',
                                   'data_split']].loc[good_timit._full_dataset_index.values]
munged_excerpt_labels.reset_index(inplace=True, drop=True)

In [17]:
good_timit.reset_index(inplace=True, drop=True)

In [18]:
new_timit = pd.concat([good_timit, munged_excerpt_labels], axis=1)

In [19]:
new_timit.columns

Index(['start_sample', 'end_sample', 'word', '_full_dataset_index', 'corpus',
       'path', 'source', 'speaker', 'sr', 'signal_length', 'duration_ms',
       'n_channels', 'word_length_samples', 'num_samples_before',
       'num_samples_after', 'start_ms', 'end_ms', 'signal', 'speaker_sex',
       'sentence_type', 'sentence_id', 'dialect_region', 'data_split'],
      dtype='object')

In [20]:
new_timit.head()

,start_sample,end_sample,word,_full_dataset_index,corpus,path,source,speaker,sr,signal_length,...,num_samples_before,num_samples_after,start_ms,end_ms,signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split
0,9190,11517,your,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,...,9190,35280,0.574375,0.719812,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
1,11517,16334,dark,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,...,11517,30463,0.719812,1.020875,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
2,16334,21199,suit,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,...,16334,25598,1.020875,1.324938,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
3,21199,22560,in,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,...,21199,24237,1.324938,1.410000,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
4,22560,28064,greasy,0,timit,/home/raygon/projects/user/jfeather/jsinDatase...,train-dr1-fcjf0-sa1,fcjf0,16000,46797,...,22560,18733,1.410000,1.754000,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train


In [21]:
## Remove unneeded columns

new_timit.drop(columns=['path', 'corpus', 'start_ms', 'end_ms'], inplace=True)

In [22]:
# rename signal column
new_timit.rename(columns={'signal':'original_signal'}, inplace=True)

In [23]:
new_timit.head()

,start_sample,end_sample,word,_full_dataset_index,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,original_signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split
0,9190,11517,your,0,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,2327,9190,35280,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
1,11517,16334,dark,0,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4817,11517,30463,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
2,16334,21199,suit,0,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,4865,16334,25598,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
3,21199,22560,in,0,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,1361,21199,24237,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train
4,22560,28064,greasy,0,train-dr1-fcjf0-sa1,fcjf0,16000,46797,2924.8125,1,5504,22560,18733,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train


In [24]:
new_timit.shape

(18055, 19)

In [25]:
# cut signals so they are all exactly 2 seconds - get endpoints based on word start and end samples 


def cut_signal(row):
    word_dur = row.end_sample - row.start_sample
    remainder = 32000 - word_dur
    edge_shift = remainder // 2
    if remainder%2 == 1: # if odd frames, just shift by one
        start_ix = row.start_sample - (edge_shift + 1) 
    else:
        start_ix = row.start_sample - edge_shift
    end_ix  = row.end_sample + edge_shift
    # need to ix original_signal[0] to get array due to wrapping in timit df's 
    sig = row.original_signal[0][start_ix:end_ix]
    return sig

new_timit['signal'] = new_timit[['start_sample', 'end_sample', 'original_signal']].apply(lambda x: cut_signal(x), axis=1)



In [26]:
new_timit.shape

(18055, 20)

#### Filter excerpts that are cut incorrectly (these have 0 length) - this is a temporary hack

In [27]:
new_timit['signal_length'] = new_timit['signal'].apply(lambda x: len(x))

In [28]:
new_timit = new_timit[new_timit['signal_length'] == 32000]

In [29]:
new_timit.reset_index(inplace=True, drop=True)

In [31]:
new_timit.shape

(10309, 20)

In [37]:
def resample(x):
    return nnresample.resample(x,5,4, axis=1) # new sr is 20kHz old is 16kHz, is 5/4 ratio 


In [40]:
upsampled_signals = resample(np.stack(new_timit['signal']))


array([ 0.00010568,  0.00012089,  0.00010269, ..., -0.00563099,
       -0.00073062,  0.00203871])

In [50]:
new_timit['signal'] = [sig for sig in upsampled_signals]

In [51]:
new_timit['signal']

0        [0.00010567984837771834, 0.0001208881458073989...
1        [-0.004511013371148214, -0.003617504427105747,...
2        [0.0004276240053091666, -0.0008603203611409375...
3        [0.0001521242010561249, 0.00016478710664698444...
4        [0.0005179703410519659, 0.001535978666334516, ...
                               ...                        
10304    [0.001070872206584041, 0.0005574565431734798, ...
10305    [-0.00930564273998756, -0.008628906170615836, ...
10306    [0.01691577257610505, 0.01807478708642801, 0.0...
10307    [-3.506950995486581e-05, -1.7627318021620925e-...
10308    [-0.0011598159157006857, -0.001120112916833320...
Name: signal, Length: 10309, dtype: object

In [52]:
new_timit['sr'] = 20000 # update sampling rate column 

In [53]:
new_timit['signal_length'] = new_timit['signal'].apply(lambda x: len(x)) # update signal length column

In [54]:
new_timit.shape

(10309, 20)

In [55]:
new_timit = new_timit[new_timit['signal_length'] == 40000].reset_index(drop=True)

In [56]:
new_timit.shape

(10309, 20)

In [57]:
new_timit.head()

,start_sample,end_sample,word,_full_dataset_index,source,speaker,sr,signal_length,duration_ms,n_channels,word_length_samples,num_samples_before,num_samples_after,original_signal,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal
0,16334,21199,suit,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,2924.8125,1,4865,16334,25598,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train,"[0.00010567984837771834, 0.0001208881458073989..."
1,21199,22560,in,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,2924.8125,1,1361,21199,24237,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train,"[-0.004511013371148214, -0.003617504427105747,..."
2,22560,28064,greasy,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,2924.8125,1,5504,22560,18733,"[[3.051850947599719e-05, -3.051850947599719e-0...",f,sa,1,dr1,train,"[0.0004276240053091666, -0.0008603203611409375..."
3,16041,18300,she,2,train-dr1-fcjf0-si1027,fcjf0,20000,40000,3091.2500,1,2259,16041,31160,"[[0.00024414807580797754, 0.000152592547379985...",f,si,1027,dr1,train,"[0.0001521242010561249, 0.00016478710664698444..."
4,18300,22500,took,2,train-dr1-fcjf0-si1027,fcjf0,20000,40000,3091.2500,1,4200,18300,26960,"[[0.00024414807580797754, 0.000152592547379985...",f,si,1027,dr1,train,"[0.0005179703410519659, 0.001535978666334516, ..."


In [60]:
Audio(new_timit['signal'].iloc[6], rate = 20000)

In [62]:
new_timit.columns

Index(['start_sample', 'end_sample', 'word', '_full_dataset_index', 'source',
       'speaker', 'sr', 'signal_length', 'duration_ms', 'n_channels',
       'word_length_samples', 'num_samples_before', 'num_samples_after',
       'original_signal', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal'],
      dtype='object')

In [64]:
# drop unnecessary set of columns 

new_timit.drop(columns=['start_sample', 'end_sample','duration_ms', 'n_channels',
       'word_length_samples', 'num_samples_before', 'num_samples_after',
       'original_signal'], inplace=True)

In [65]:
new_timit.head()

,word,_full_dataset_index,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal
0,suit,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,f,sa,1,dr1,train,"[0.00010567984837771834, 0.0001208881458073989..."
1,in,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,f,sa,1,dr1,train,"[-0.004511013371148214, -0.003617504427105747,..."
2,greasy,0,train-dr1-fcjf0-sa1,fcjf0,20000,40000,f,sa,1,dr1,train,"[0.0004276240053091666, -0.0008603203611409375..."
3,she,2,train-dr1-fcjf0-si1027,fcjf0,20000,40000,f,si,1027,dr1,train,"[0.0001521242010561249, 0.00016478710664698444..."
4,took,2,train-dr1-fcjf0-si1027,fcjf0,20000,40000,f,si,1027,dr1,train,"[0.0005179703410519659, 0.001535978666334516, ..."


In [66]:
new_timit.shape

(10309, 12)

### Add word ix labels from our vocabulary

In [99]:
word_to_ix_map = {word:int(ix) for ix, word in class_map.items()}

In [100]:
new_timit['word_int'] = new_timit['word'].map(word_to_ix_map)

In [101]:
new_timit['word_int'].isnull().values.any() 

True

In [108]:
new_timit['word_int'] = new_timit['word_int'].replace(np.nan, 0)

In [110]:
new_timit['word_int'] = pd.to_numeric(new_timit['word_int'], downcast='integer')

In [114]:
new_timit['word_int']

0

### Save dataframe 


In [2]:
out_path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'

In [3]:
from pathlib import Path

In [4]:
path = Path(out_path)

In [5]:
path

PosixPath('/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl')

In [6]:
# new_timit.to_pickle(path)

In [7]:
new_timit = pd.read_pickle(path)

### Get speakers with more than one occurance 

This is needed to obtain cue

In [10]:
len(new_timit.word_int.unique())

319

In [19]:
new_timit.word_int.value_counts()[new_timit.word_int.value_counts() <= ]

78     10
142    10
320    10
7      10
434    10
       ..
251     1
231     1
502     1
149     1
333     1
Name: word_int, Length: 311, dtype: int64

In [46]:
counts = new_timit['speaker'].value_counts()

new_timit = new_timit[new_timit.speaker.isin(counts.index[counts.gt(1)])]

In [126]:
target_timit = new_timit[new_timit['word_int'] !=0]
rest_timit = new_timit[new_timit['word_int'] ==0]

In [129]:
talkers = target_timit.speaker.unique()

all([len(rest_timit[rest_timit['speaker'] == talker]) > 0 for talker in talkers])

True

In [49]:
new_timit['speaker'].value_counts().min()


2

In [50]:
new_timit.reset_index(inplace=True, drop=True)

In [5]:
new_timit = pd.read_pickle(path)

In [6]:
new_timit.head()

,word,_full_dataset_index,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,data_split,signal,word_int
0,programs,15,train-dr1-fdaw0-sx146,fdaw0,20000,40000,f,sx,146,dr1,train,"[2.3069690259136608e-05, 0.0001174022678404248...",552
1,novel,17,train-dr1-fdaw0-sx326,fdaw0,20000,40000,f,sx,326,dr1,train,"[0.0010651568947912448, 0.007388919911868498, ...",461
2,social,35,train-dr1-fecd0-sx158,fecd0,20000,40000,f,sx,158,dr1,train,"[-9.94791987722646e-05, -0.0001146543672654921...",659
3,light,36,train-dr1-fecd0-sx248,fecd0,20000,40000,f,sx,248,dr1,train,"[0.005607189393168457, 0.003494775608541276, 0...",393
4,status,39,train-dr1-fecd0-sx68,fecd0,20000,40000,f,sx,68,dr1,train,"[-0.0008464058035127516, 0.0002213455216032013...",683


In [8]:
len(new_timit.speaker.unique())

266

### Develop `__getitem__` for pytorch dataset class 

In [72]:
index = 10

In [75]:
foreground = new_timit.signal[2]

KeyError: 2

In [ ]:
talker = new_timit.speaker[index]

In [ ]:
talker

In [ ]:
cue_ixs = np.where(new_timit.speaker == talker)[0]

In [ ]:
cue_ixs

In [ ]:
cue_ixs[cue_ixs != index]

In [ ]:
cue_ix = np.random.choice(cue_ixs)

In [ ]:
np.random.choice(cue_ixs)

In [ ]:
fg_cue = new_timit.signal[cue_ix]

In [ ]:
bg_ixs = np.where(new_timit.speaker != talker)[0]

#### Test single bg talker selection

In [ ]:
talker_ixs = np.random.choice(bg_ixs, size=1, replace=False) # returns array

In [ ]:
background_talkers = new_timit.signal[talker_ixs]
background_talkers = np.stack(background_talkers)
print(background_talkers.shape)

In [ ]:
background_talkers.squeeze().shape

#### Test multi-bg talker selection

In [ ]:
talker_ixs = np.random.choice(bg_ixs, size=4, replace=False)


In [ ]:
talker_ixs

In [ ]:
background_talkers = new_timit.signal[talker_ixs]
np.stack(background_talkers).shape

#### Most general:  Just use stack and strip singleton dimension in 1 bg talker case

In [ ]:
new_timit['speaker']